# 01_03 How a program learns

*📖 Book: appendix A*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **the basics** (chapter 1): the ideas every stage uses
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** A new model's weights are random. How does a program change them, by itself, until its guesses are good?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- The learning loop: guess → measure the mistake → adjust → repeat
- How to learn one weight by hand
- How PyTorch finds the direction to adjust: gradients and backpropagation
- Where this loop sits in our training code

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **loss** | a mistake score: one number for how wrong the guesses were; lower is better | `(guess - answer)²` |
| **gradient** | for each weight: which way, and how steeply, the loss goes up if that weight grows; we step the other way | `w.grad` = `-27.0` |
| **backpropagation** | the method PyTorch uses to work out every weight's gradient, going backwards from the loss | `loss.backward()` |
| **optimizer** | the PyTorch tool that adjusts the weights using their gradients | `torch.optim.SGD` |

## 3. The idea

### ❓ What does it mean for a program to learn?

Its weights change so that its guesses get better. It happens in a loop:

```
guess → measure the mistake → adjust the weights a little → repeat
```

Nobody types in the right weights by hand. The program finds them itself, one small adjustment at a time.

### ❓ How do we measure the mistake?

With a **loss**: one number that's big when the guesses are bad and small when they're good. A simple one is the mistake multiplied by itself, `(guess - answer)²`. Squaring makes it positive either way, so too high and too low both count as wrong.

### ❓ How does it know which way to adjust?

Picture yourself on a hillside in fog, where your height is the loss. You can't see the bottom, but you can feel which way the ground slopes. So you take a small step downhill, and then another. That slope, for each weight, is its **gradient**.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Can I learn one weight by hand?

Let's try. The task is to find `w` so that `w × 3` gives `6`. (It's 2, obviously, but the program doesn't know that.) We start at `w = 0.5`. Each round we guess, see how far off we are, and move `w` a little in the right direction:

```
w       guess = w × 3    mistake = guess − 6    nudge = −0.1 × mistake
0.500   1.500            −4.500                 +0.450
0.950   2.850            −3.150                 +0.315
1.265   …
```

The guess is too small, so the mistake is negative, so the nudge pushes `w` up.

👉 **Predict first:** will `w` get to 2 in one step, or creep towards it?

In [ ]:
w = 0.5
for step in range(8):
    guess = w * 3
    mistake = guess - 6  # negative = guess too small
    print(f"step {step}: w = {w:.3f}, guess = {guess:.3f}, mistake = {mistake:.3f}")
    w = w - 0.1 * mistake  # nudge w against the mistake

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`w` creeps up from 0.5 (0.950, 1.265, 1.486, …) and the mistake shrinks every time. That `0.1` is the size of the nudge. Too big and you overshoot, too small and it takes forever. Its proper name is the **learning rate**, and [05_02](../05_training_on_the_odyssey/05_02_learning_rate.ipynb) is all about it.

</details>

### ❓ Your turn: can you do one nudge yourself?

Same task (`w × 3` should give `6`), but this time we start at `w = 1.0`. The guess and the mistake are worked out for you. You write the nudge.

👉 **Predict first:** the guess is 3, which is too small. Should the new `w` be above or below 1.0?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
start_w = 1.0
guess = start_w * 3
mistake = guess - 6  # negative = guess too small

# 👉 do: nudge start_w against the mistake, with a nudge size of 0.1
new_w = None

In [ ]:
from homer_ai.lessons import check

check(
    "w after one nudge",
    new_w,
    1.3,
    coach="Got 0.7? Then you added the mistake. The mistake is negative here "
    "(the guess is too small), so we subtract it, and that pushes w up.",
)

<details><summary>💡 Hint</summary>

It's the last line of the loop above, `w = w - 0.1 * mistake`, with `start_w` in place of `w`.

</details>

<details><summary>✅ Answer</summary>

```python
start_w = 1.0
guess = start_w * 3
mistake = guess - 6  # negative = guess too small

new_w = start_w - 0.1 * mistake
```

The mistake is `3 − 6 = −3`, so the nudge is `−0.1 × −3 = +0.3` and `w` goes from 1.0 to 1.3. Closer to 2, but not there yet.

</details>

### ❓ Can I see it learn?

Sure. We run the same loop again, but this time we write down `w` and the loss at every step, then draw them. The dashed line is the answer, `2`.

👉 **Predict first:** will `w` bend smoothly onto the dashed line, or zigzag across it?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, GOOD, INK, MUTED, studio_axes, studio_figure

w = 0.5
ws, losses = [], []
for _step in range(15):
    ws.append(w)
    losses.append((w * 3 - 6) ** 2)  # the loss: (guess - answer) squared
    w = w - 0.1 * (w * 3 - 6)

fig = studio_figure("One weight, learning", f"w creeps from {ws[0]} to {ws[-1]:.2f}")
left = studio_axes(fig, (0.09, 0.12, 0.38, 0.68))
left.plot(ws, color=ACCENT, linewidth=3, marker="o", markersize=5)
left.axhline(2, color=GOOD, linestyle="--", linewidth=1.5)
left.text(14, 2.06, "target 2", color=GOOD, ha="right", family="monospace")
left.set_xlabel("step")
left.set_ylabel("w")

right = studio_axes(fig, (0.57, 0.12, 0.38, 0.68))
right.plot(losses, color=INK, linewidth=3, marker="o", markersize=5)
right.set_xlabel("step")
right.set_ylabel("loss")
right.text(0.5, losses[0], f"{losses[0]:.1f}", color=MUTED, family="monospace")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`w` bends smoothly onto the dashed line, fast at first, then slower as the mistake shrinks. The loss falls the same way: steep at the start, flat at the end. That flat end is what a program that has learned looks like.

</details>

### ❓ How does PyTorch do the same thing?

With three calls that you'll see in every training loop:

- `loss.backward()` works out the gradient of every weight. The method is called **backpropagation**, because it goes backwards from the loss through every calculation.
- `optimizer.step()` nudges every weight downhill, using its gradient. **SGD** (stochastic gradient descent, or "step down the slope") is the simplest optimizer: weight = weight − nudge size × gradient.
- `optimizer.zero_grad()` wipes the last round's gradients first, like clearing a calculator. Otherwise they'd pile up.

`requires_grad=True` tells PyTorch to keep track of `w` so it can find its gradient.

👉 **Predict first:** at `w = 0.5`, the guess is too small. Will the gradient be positive or negative?

In [ ]:
import torch

w = torch.tensor(0.5, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.02)

for step in range(8):
    loss = (w * 3 - 6) ** 2
    optimizer.zero_grad()
    loss.backward()
    grad = w.grad.item()
    w_now, loss_now = w.item(), loss.item()
    print(f"step {step}: w = {w_now:.4f}, loss = {loss_now:.4f}, gradient = {grad:.3f}")
    optimizer.step()

print(f"final w = {w.item():.4f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The gradient starts at **−27.0**. Negative means "growing `w` makes the loss go down", so the step makes `w` bigger. The loss falls 20.25 → 8.29 → 3.40 → … → 0.04, and `w` ends up at about 1.958, close to 2.

Where does −27 come from? The by-hand loop moved `w` by `0.1 × mistake`. The gradient of this loss is bigger than the mistake: it's `2 × 3 × mistake` (the 2 comes from the square, the 3 from `w × 3`). At `w = 0.5` the mistake is −4.5, so the gradient is `6 × −4.5 = −27`. SGD moves `w` by `lr × gradient`, which is `lr × 6 × mistake`. So `lr=0.02` here is a nudge of `0.12 × mistake`, about the same size as the `0.1 × mistake` we used by hand. The two 0.1s you'll see are not the same thing.

</details>

## 5. Full example

### ❓ Where is this loop in our real training code?

`train()` in `src/homer_ai/train_homer.py` does exactly the same four things, only with 11,840 weights instead of one:

```python
logits = model(inputs)                    # 1. guess
loss = F.cross_entropy(logits, targets)   # 2. measure the mistake
optimizer.zero_grad()                     #    clear the last gradients
loss.backward()                           # 3. find every weight's gradient
optimizer.step()                          # 4. nudge every weight downhill
```

`logits` is just the usual name for the model's raw scores. `cross_entropy` is a loss made for guessing between many options (chapter 5). And our optimizer is `AdamW`, a smarter one that adapts the step size for each weight.

One last experiment with our single weight. We raise the learning rate from 0.02 to `lr=0.1`. From the sum above, that's a nudge of `0.1 × 6 × mistake = 0.6 × mistake`, six times the by-hand one.

👉 **Predict first:** with this bigger nudge on the same task, do you get closer to 2 in 8 steps, or overshoot?

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.1)

for step in range(8):
    loss = (w * 3 - 6) ** 2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"step {step}: w = {w.item():.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`w` jumps right past 2, to 3.20, then swings back and forth around it (1.04, 2.77, 1.39, …). After 8 steps it's at 1.75, which is further from 2 than the smaller nudge got (1.958). Every step overshoots. (By hand, `0.1` crept up smoothly. Here `lr=0.1` overshoots, because SGD multiplies it by the gradient, which is 6 times the mistake.) So picking the nudge size matters, and chapter 5 shows how to do it.

</details>

## 6. Recap

- Learning is a loop: guess → loss → adjust → repeat.
- The gradient says which way is downhill; backpropagation (`loss.backward()`) finds it for every weight.
- `optimizer.step()` takes the step; the nudge size is the learning rate.

**You can now:**

- nudge one weight against its mistake, by hand
- write the PyTorch training loop: `zero_grad`, `backward`, `step`
- tell when a learning rate is too big: the weight jumps past the answer

**Next:** [01_04: Our model as a black box](01_04_our_model_as_a_black_box.ipynb).